- **학습 목표**: RSS 피드를 파싱해 실제 기사 목록(제목·요약·링크)을 구조화된 형태로 받아올 수 있다.

In [1]:
import feedparser

In [4]:
def fetch_articles(rss_url: str, limit: int = 5) -> list[dict]:
    """
    요구사항:
    - feedparser.parse(rss_url)로 피드를 가져온다.
    - feed.bozo가 True면 (파싱 에러) 경고를 출력한다.
    - entries 중 상위 limit개에서 title, summary(없으면 빈 문자열), link를 추출한다.
    - 반환값: [{"title": ..., "summary": ..., "link": ...}, ...]
    """
    result = []
    feed = feedparser.parse(rss_url)
    limit = min(limit, len(feed.entries))
    if feed.bozo:
        print(f"Warning: Failed to parse RSS feed from {rss_url}.")
    for i in range(0,limit):
        entry = feed.entries[i]
        result.append({"title": entry.title, "summary": entry.get("summary", ""), "link": entry.link})
    return result
fetch_articles("https://www.yna.co.kr/rss/industry.xml", 5)

[{'title': '[인사] 화승',
  'summary': '◇ 전무 승진',
  'link': 'https://www.yna.co.kr/view/AKR20260901028100051'},
 {'title': 'E1, 9월 LPG 국내 공급가격 동결…상업용 프로판 ㎏당 1천483원',
  'summary': '(서울=연합뉴스) 강태우 기자 = E1이 9월 액화석유가스(LPG) 공급 가격을 동결했다.',
  'link': 'https://www.yna.co.kr/view/AKR20260901027200003'},
 {'title': '한투증권 "9월 투자 전략, IT 대형주에 시클리컬 업종 추가"',
  'summary': '(서울=연합뉴스) 임은진 기자 = 한국투자증권은 9월 투자 전략으로 IT 대형주에 실적 개선이 확인되는 시클리컬(경기 순환) 업종을 더하는 포트...',
  'link': 'https://www.yna.co.kr/view/AKR20260901025800008'},
 {'title': '경기도, 소·염소 47만 마리 구제역 백신 접종',
  'summary': '(의정부=연합뉴스) 김도윤 기자 = 경기도는 1일부터 한 달간 지역 내 소와 염소 약 47만 마리를 대상으로 O+A형 구제역 백신을 접종한다.',
  'link': 'https://www.yna.co.kr/view/AKR20260901025300060'},
 {'title': '부산시, AI기반 지능형 사이버 보안관제 가동…변종공격 차단',
  'summary': '(부산=연합뉴스) 김선호 기자 = 부산시는 1일부터 &apos;인공지능(AI) 기반 지능형 사이버 보안관제 시스템&apos; 운영을 시작한다.',
  'link': 'https://www.yna.co.kr/view/AKR20260901022100051'}]

In [ ]:
- **학습 목표**: Day1의 단일 호출 함수를 여러 기사에 반복 적용하는 배치 파이프라인으로 확장하고, API rate limit을 고려한다.

In [8]:
import openai
def summarize_and_classify(client, title: str, text: str, categories: list[str]) -> dict:

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that summarizes text and classifies it into one of the provided categories. "
                    "Please respond in the following format:\n"
                    "그 외의 설명, 인사말, 마크다운은 무시하고, 반드시 아래 형식으로만 답변\n"
                    "요약: <2문장 요약>\n"
                    "카테고리: <categories 중 하나>"
                ),
            },
            {
                "role": "user",
                "content": f"Title: {title}\nText: {text}\nCategories: {', '.join(categories)}",
            },
        ])
    response_text = response.choices[0].message.content
    response_lines = response_text.splitlines()
    summary = None
    category = "Other"  # Default fallback category
    for line in response_lines:
        line = line.strip()
        if line.startswith("요약:"):
            summary = line[len("요약:"):].strip()
        elif line.startswith("카테고리:"):
            category = line[len("카테고리:"):].strip()
    return {"title": title, "summary": summary, "category": category}
client = openai.OpenAI()  # Initialize your client here
title = "Sample Title"
text = "This is a sample text that needs to be summarized and classified into a category."
categories = ["Technology", "Science", "Health", "Entertainment"]
summarize_and_classify(client, title, text, categories)

{'title': 'Sample Title',
 'summary': '이 텍스트는 요약되고 분류되어야 하는 샘플 텍스트입니다. 특정한 주제나 내용이 나타나지 않아 일반적인 설명으로 보입니다.',
 'category': '그 외의 설명'}

In [13]:

import time

In [15]:
def process_feed(client, rss_url: str, categories: list[str], limit: int = 5) -> list[dict]:
    """
    요구사항:
    - fetch_articles()로 기사를 가져온다.
    - 각 기사에 대해 summarize_and_classify()를 호출한다.
    - 각 호출 사이에 time.sleep(1) 등으로 rate limit 여유를 둔다.
    - 처리하면서 "[카테고리] 제목 → 요약"을 즉시 출력한다(진행 상황 확인용).
    - 반환값: 각 기사의 결과 dict 리스트.
    """
    articles = fetch_articles(rss_url, limit)
    for article in articles:
        content = summarize_and_classify(client, article["title"], article["summary"], categories)
        print(f"[{content['category']}] {content['title']} → {content['summary']}")
        time.sleep(1)  # Rate limit 여유
    return articles
process_feed(client, "https://www.yna.co.kr/rss/industry.xml", ["Technology", "Science", "Health", "Entertainment"], 1)

[Health] [부산소식] 외국인 환자 유치 비즈니스 페어 → 부산경제진흥원이 외국인 환자 유치와 의료관광 네트워크 강화를 위한 비즈니스 페어를 개최한다. 이는 부산의 의료관광 산업 발전을 목표로 한다.


[{'title': '[부산소식] 외국인 환자 유치 비즈니스 페어',
  'summary': '(부산=연합뉴스) ▲ 외국인 환자 유치 비즈니스 페어 = 부산경제진흥원은 외국인 환자 유치 활성화와 부산 의료관광의 해외 네트워크 강화를 위해 ...',
  'link': 'https://www.yna.co.kr/view/AKR20260901039000051'}]